Apriori Algorithmn

In [2]:
from collections import defaultdict

def min_supp_count(min_supp, total_trans):
    count = min_supp * total_trans  # min_supp is percentage (e.g., 0.05 for 5%)
    return count

def get_frequent_itemsets(data, min_supp_count_value):

    symptom_counts = defaultdict(int)

    for i in data: #loop throgh the list and count
        for j in i:
            symptom_counts[j] += 1

    # Filter by minimum support
    frequent = {}
    for i in symptom_counts:
        count = symptom_counts[i]
        if count >= min_supp_count_value:
            frequent[frozenset([i])] = count


    list_for_rule_generation = frequent.copy() #make a copy to store all frequent items
    templist = frequent
    k = 2

    print(f"Found {len(frequent)} frequent 1 itemsets")

    #Generate larger itemsets
    while templist:
        print(f"Generating {k} itemsets")

        # Create candidate itemsets
        candidates = set()
        itemsets_list = list(templist.keys())

        # Combine itemsets
        for i in range(len(itemsets_list)):
            for j in range(i + 1, len(itemsets_list)):
                itemset1 = itemsets_list[i]
                itemset2 = itemsets_list[j]

                # Check if 2 itemsets have something in common then fuse them
                union_set = itemset1.union(itemset2)
                if len(union_set) == k:
                    candidates.add(union_set)

        if not candidates: #if nothing to combine then exits
            break

        # Count support
        candidate_support = {}
        for candidate in candidates:
            count = 0
            for transaction in data:
                if candidate.issubset(set(transaction)):
                    count += 1

            if count >= min_supp_count_value:
                candidate_support[candidate] = count

        if not candidate_support:
            break

        print(f"Found {len(candidate_support)} frequent {k}-itemsets")
        list_for_rule_generation.update(candidate_support)
        templist = candidate_support
        k += 1

    return list_for_rule_generation

def generate_association_rules(frequent_itemsets, min_confidence, total_transactions):

    rules = []

    for itemset, support_count in frequent_itemsets.items():
        if len(itemset) >= 2:  # generate rules for itemsets with at least 2 items
            items_list = list(itemset)

            # Generate all possible rules for this itemset
            # For itemset {A,B,C}, generate rules eg a-bc, b-ac ....
            from itertools import combinations

            # Generate all non-empty proper subsets on the left
            for r in range(1, len(items_list)):
                for antecedent_items in combinations(items_list, r):
                    antecedent_set = frozenset(antecedent_items)
                    consequent_set = itemset - antecedent_set

                    # Calculate confidence
                    if antecedent_set in frequent_itemsets:
                        antecedent_support = frequent_itemsets[antecedent_set] #(total set count)/(left set count)
                        confidence = support_count / antecedent_support

                        if confidence >= min_confidence: #add into rules if it is >= to the minconfidence
                            rules.append({
                                'rule_id': len(rules) + 1,
                                'antecedent': antecedent_set, #left
                                'consequent': consequent_set, #right
                                'support': support_count / total_transactions,
                                'support_count': support_count,
                                'confidence': confidence
                            })

    return rules

def apriori2(data, min_supp, confidence):

    total_count = len(data)
    min_supp_count_value = min_supp_count(min_supp, total_count)

    print(f"Apriori")
    print(f"Total transactions: {total_count}")
    print(f"Minimum support: {min_supp:.1%} (≥{min_supp_count_value})")
    print(f"Minimum confidence: {confidence:.1%}")

    frequent_itemsets = get_frequent_itemsets(data, min_supp_count_value)

    itemset_sizes = {}
    for itemset in frequent_itemsets.keys():
        size = len(itemset)
        itemset_sizes[size] = itemset_sizes.get(size, 0) + 1

    for size, count in sorted(itemset_sizes.items()):
        print(f"  {size}-itemsets: {count}")

    print(f"\nSTEP 2: GENERATING ASSOCIATION RULES...")
    rules = generate_association_rules(frequent_itemsets, confidence, total_count)

    print(f"Generated {len(rules)} association rules")

    return frequent_itemsets, rules

Import Dataset

In [3]:
import os
import pandas as pd

import pandas as pd
import os

# Load the dataset.csv file
df = pd.read_csv(r'C:\Users\rqpua\Downloads\dataset.csv')
df.head()

,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,Fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Fungal infection,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Fungal infection,itching,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Fungal infection,itching,skin_rash,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Fungal infection,itching,skin_rash,nodal_skin_eruptions,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df.head()

,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,Fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Fungal infection,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Fungal infection,itching,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Fungal infection,itching,skin_rash,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Fungal infection,itching,skin_rash,nodal_skin_eruptions,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
## Data Cleaning step

df.fillna(0,inplace=True)

In [6]:
df.head()

,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,Fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic _patches,0,0,0,0,0,0,0,0,0,0,0,0,0
1,Fungal infection,skin_rash,nodal_skin_eruptions,dischromic _patches,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,Fungal infection,itching,nodal_skin_eruptions,dischromic _patches,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,Fungal infection,itching,skin_rash,dischromic _patches,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,Fungal infection,itching,skin_rash,nodal_skin_eruptions,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [7]:
#See all the symptoms
symptom_columns = [col for col in df.columns if col.startswith('Symptom_')]

#Get unique ones
all_symptoms = df[symptom_columns].values.flatten()
unique_symptoms = pd.unique(all_symptoms)

# Remove any NaN values
unique_symptoms = unique_symptoms[~pd.isna(unique_symptoms)]

print(f"Found {len(unique_symptoms)} unique symptoms:")
print(unique_symptoms)

Found 132 unique symptoms:
['itching' ' skin_rash' ' nodal_skin_eruptions' ' dischromic _patches' 0
 ' continuous_sneezing' ' shivering' ' chills' ' watering_from_eyes'
 ' stomach_pain' ' acidity' ' ulcers_on_tongue' ' vomiting' ' cough'
 ' chest_pain' ' yellowish_skin' ' nausea' ' loss_of_appetite'
 ' abdominal_pain' ' yellowing_of_eyes' ' burning_micturition'
 ' spotting_ urination' ' passage_of_gases' ' internal_itching'
 ' indigestion' ' muscle_wasting' ' patches_in_throat' ' high_fever'
 ' extra_marital_contacts' ' fatigue' ' weight_loss' ' restlessness'
 ' lethargy' ' irregular_sugar_level' ' blurred_and_distorted_vision'
 ' obesity' ' excessive_hunger' ' increased_appetite' ' polyuria'
 ' sunken_eyes' ' dehydration' ' diarrhoea' ' breathlessness'
 ' family_history' ' mucoid_sputum' ' headache' ' dizziness'
 ' loss_of_balance' ' lack_of_concentration' ' stiff_neck' ' depression'
 ' irritability' ' visual_disturbances' ' back_pain' ' weakness_in_limbs'
 ' neck_pain' ' weakness_of_

In [8]:
# Clean the data 
df_clean = df.copy()
symptom_cols = [col for col in df.columns if col.startswith('Symptom_')]

#Clean the symptom strings 
for col in symptom_cols:
    df_clean[col] = df_clean[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

#Apply standardization for names of similiar meaning
symptom_mapping = {
    'abdominal_pain': 'stomach_pain',
    'belly_pain': 'stomach_pain'
}

for col in symptom_cols:
    df_clean[col] = df_clean[col].replace(symptom_mapping)

#Get unique symptoms and clean them
all_symptoms_clean = df_clean[symptom_cols].values.flatten()
unique_symptoms_clean = pd.unique(all_symptoms_clean)

# Filter out NaN, 0, and empty values, then convert to strings
unique_symptoms_clean = [str(s).strip() for s in unique_symptoms_clean 
                        if not pd.isna(s) and s != 0 and s != '0' and str(s).strip() != '']

# Remove duplicates after cleaning
unique_symptoms_clean = list(set(unique_symptoms_clean))

print(f"After proper standardization: {len(unique_symptoms_clean)} unique symptoms")
print("Standardized symptoms:")
for symptom in sorted(unique_symptoms_clean):
    print(f"- {symptom}")

After proper standardization: 129 unique symptoms
Standardized symptoms:
- abnormal_menstruation
- acidity
- acute_liver_failure
- altered_sensorium
- anxiety
- back_pain
- blackheads
- bladder_discomfort
- blister
- blood_in_sputum
- bloody_stool
- blurred_and_distorted_vision
- breathlessness
- brittle_nails
- bruising
- burning_micturition
- chest_pain
- chills
- cold_hands_and_feets
- coma
- congestion
- constipation
- continuous_feel_of_urine
- continuous_sneezing
- cough
- cramps
- dark_urine
- dehydration
- depression
- diarrhoea
- dischromic _patches
- distention_of_abdomen
- dizziness
- drying_and_tingling_lips
- enlarged_thyroid
- excessive_hunger
- extra_marital_contacts
- family_history
- fast_heart_rate
- fatigue
- fluid_overload
- foul_smell_of urine
- headache
- high_fever
- hip_joint_pain
- history_of_alcohol_consumption
- increased_appetite
- indigestion
- inflammatory_nails
- internal_itching
- irregular_sugar_level
- irritability
- irritation_in_anus
- itching
- join

In [9]:
df_clean.head(4920)

,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,Fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic _patches,0,0,0,0,0,0,0,0,0,0,0,0,0
1,Fungal infection,skin_rash,nodal_skin_eruptions,dischromic _patches,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,Fungal infection,itching,nodal_skin_eruptions,dischromic _patches,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,Fungal infection,itching,skin_rash,dischromic _patches,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,Fungal infection,itching,skin_rash,nodal_skin_eruptions,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4915,(vertigo) Paroymsal Positional Vertigo,vomiting,headache,nausea,spinning_movements,loss_of_balance,unsteadiness,0,0,0,0,0,0,0,0,0,0,0
4916,Acne,skin_rash,pus_filled_pimples,blackheads,scurring,0,0,0,0,0,0,0,0,0,0,0,0,0
4917,Urinary tract infection,burning_micturition,bladder_discomfort,foul_smell_of urine,continuous_feel_of_urine,0,0,0,0,0,0,0,0,0,0,0,0,0
4918,Psoriasis,skin_rash,joint_pain,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,0,0,0,0,0,0,0,0,0,0,0


In [10]:
# Make each row as a basket and remove disease 
symptom_cols = [col for col in df_clean.columns if col.startswith('Symptom_')]

transactions_clean = []
for i in range(len(df_clean)):
    transaction = []
    for col in symptom_cols:
        symptom_value = df_clean.loc[i, col]
        if str(symptom_value) != '0' and not pd.isna(symptom_value) and str(symptom_value).strip() != '':
            transaction.append(str(symptom_value).strip())
    transactions_clean.append(transaction)

print(f"Created {len(transactions_clean)} cleaned transactions")
print("First 5 cleaned transactions:")
for i in range(5):
    print(f"{df_clean.loc[i, 'Disease']}: {transactions_clean[i]}")

Created 4920 cleaned transactions
First 5 cleaned transactions:
Fungal infection: ['itching', 'skin_rash', 'nodal_skin_eruptions', 'dischromic _patches']
Fungal infection: ['skin_rash', 'nodal_skin_eruptions', 'dischromic _patches']
Fungal infection: ['itching', 'nodal_skin_eruptions', 'dischromic _patches']
Fungal infection: ['itching', 'skin_rash', 'dischromic _patches']
Fungal infection: ['itching', 'skin_rash', 'nodal_skin_eruptions']


In [11]:
# Run Apriori
min_support = 0.15  # 15% minimum support
min_confidence = 0.5  # 50% minimum confidence

frequent_itemsets, association_rules = apriori2(transactions_clean, min_support, min_confidence)


Apriori
Total transactions: 4920
Minimum support: 15.0% (≥738.0)
Minimum confidence: 50.0%
Found 11 frequent 1 itemsets
Generating 2 itemsets
Found 8 frequent 2-itemsets
Generating 3 itemsets
  1-itemsets: 11
  2-itemsets: 8

STEP 2: GENERATING ASSOCIATION RULES...
Generated 12 association rules


In [13]:
# Display only 2-itemsets or greater
print("ALL ITEMSETS:")
print("=" * 80)
print(f"{'2-Itemset':<50} {'Support Count':<15} {'Support %':<10}")

total_transactions = len(transactions_clean)

# Only show itemsets > 2
two_itemsets = {itemset: count for itemset, count in frequent_itemsets.items() if len(itemset) >= 2}
two_itemsets_sorted = sorted(two_itemsets.items(), key=lambda x: x[1], reverse=True)

for itemset, support_count in two_itemsets_sorted:
    support_percentage = (support_count / total_transactions) * 100
    itemset_str = str(tuple(itemset))
    print(f"{itemset_str:<50} {support_count:<15} {support_percentage:.1f}%")

print(f"\nTotal itemsets found: {len(two_itemsets)}")

ALL ITEMSETS:
2-Itemset                                          Support Count   Support % 
('vomiting', 'stomach_pain')                       978             19.9%
('high_fever', 'fatigue')                          978             19.9%
('vomiting', 'nausea')                             978             19.9%
('loss_of_appetite', 'yellowing_of_eyes')          786             16.0%
('fatigue', 'loss_of_appetite')                    774             15.7%
('vomiting', 'loss_of_appetite')                   768             15.6%
('fatigue', 'vomiting')                            762             15.5%
('yellowish_skin', 'stomach_pain')                 762             15.5%

Total itemsets found: 8


In [23]:
association_rules

[{'rule_id': 1,
  'antecedent': frozenset({'vomiting'}),
  'consequent': frozenset({'stomach_pain'}),
  'support': 0.19878048780487806,
  'support_count': 978,
  'confidence': 0.5109717868338558},
 {'rule_id': 2,
  'antecedent': frozenset({'stomach_pain'}),
  'consequent': frozenset({'vomiting'}),
  'support': 0.19878048780487806,
  'support_count': 978,
  'confidence': 0.7149122807017544},
 {'rule_id': 3,
  'antecedent': frozenset({'vomiting'}),
  'consequent': frozenset({'nausea'}),
  'support': 0.19878048780487806,
  'support_count': 978,
  'confidence': 0.5109717868338558},
 {'rule_id': 4,
  'antecedent': frozenset({'nausea'}),
  'consequent': frozenset({'vomiting'}),
  'support': 0.19878048780487806,
  'support_count': 978,
  'confidence': 0.8534031413612565},
 {'rule_id': 5,
  'antecedent': frozenset({'high_fever'}),
  'consequent': frozenset({'fatigue'}),
  'support': 0.19878048780487806,
  'support_count': 978,
  'confidence': 0.7180616740088106},
 {'rule_id': 6,
  'antecedent'

In [36]:
# Convert to DataFrame 
rules_df = pd.DataFrame(association_rules)

rules_df

,rule_id,antecedent,consequent,support,support_count,confidence
0,1,(vomiting),(stomach_pain),0.198780,978,0.510972
1,2,(stomach_pain),(vomiting),0.198780,978,0.714912
2,3,(vomiting),(nausea),0.198780,978,0.510972
3,4,(nausea),(vomiting),0.198780,978,0.853403
4,5,(high_fever),(fatigue),0.198780,978,0.718062
5,6,(fatigue),(high_fever),0.198780,978,0.506211
6,7,(loss_of_appetite),(yellowing_of_eyes),0.159756,786,0.682292
7,8,(yellowing_of_eyes),(loss_of_appetite),0.159756,786,0.963235
8,9,(loss_of_appetite),(vomiting),0.156098,768,0.666667
9,10,(yellowish_skin),(stomach_pain),0.154878,762,0.835526


In [44]:
listofitem = rules_df[['antecedent', 'consequent']]
listofitem

,antecedent,consequent
0,(vomiting),(stomach_pain)
1,(stomach_pain),(vomiting)
2,(vomiting),(nausea)
3,(nausea),(vomiting)
4,(high_fever),(fatigue)
5,(fatigue),(high_fever)
6,(loss_of_appetite),(yellowing_of_eyes)
7,(yellowing_of_eyes),(loss_of_appetite)
8,(loss_of_appetite),(vomiting)
9,(yellowish_skin),(stomach_pain)


In [51]:
def get_unique_itemsets_proper(df):
    """Properly remove duplicate itemsets using frozenset"""
    seen = set()
    unique_itemsets = []
    
    for _, row in df.iterrows():
        # Create frozenset (unordered, so {x,y} == {y,x})
        itemset = frozenset([row['antecedent'], row['consequent']])
        
        if itemset not in seen:
            seen.add(itemset)
            # Convert to sorted list for consistent display
            sorted_items = sorted(list(itemset))
            unique_itemsets.append(f"{sorted_items[0]} & {sorted_items[1]}")
    
    return unique_itemsets

# Get unique itemsets
unique_itemsets = get_unique_itemsets_proper(listofitem)
print("Unique Itemsets:")
unique_itemsets

Unique Itemsets:


["frozenset({'stomach_pain'}) & frozenset({'vomiting'})",
 "frozenset({'vomiting'}) & frozenset({'nausea'})",
 "frozenset({'high_fever'}) & frozenset({'fatigue'})",
 "frozenset({'yellowing_of_eyes'}) & frozenset({'loss_of_appetite'})",
 "frozenset({'vomiting'}) & frozenset({'loss_of_appetite'})",
 "frozenset({'stomach_pain'}) & frozenset({'yellowish_skin'})",
 "frozenset({'loss_of_appetite'}) & frozenset({'fatigue'})"]

In [57]:
import re


def extract_from_string(itemset_strings):
    unique_itemsets = set()
    
    for itemset_str in itemset_strings:
        # Use regex to extract the content between quotes
        matches = re.findall(r"'([^']*)'", itemset_str)
        
        if len(matches) >= 2:
            item1, item2 = matches[0], matches[1]
            sorted_pair = tuple(sorted([item1, item2]))
            unique_itemsets.add(sorted_pair)
        else:
            print(f"Could not parse: {itemset_str}")
    
    return unique_itemsets


unique_itemsets = extract_from_string(clean_itemsets) 
print("Clean Unique Itemsets:")
unique_itemsets

Clean Unique Itemsets:


{('fatigue', 'high_fever'),
 ('fatigue', 'loss_of_appetite'),
 ('loss_of_appetite', 'vomiting'),
 ('loss_of_appetite', 'yellowing_of_eyes'),
 ('nausea', 'vomiting'),
 ('stomach_pain', 'vomiting'),
 ('stomach_pain', 'yellowish_skin')}

In [59]:
df_clean

,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,Fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic _patches,0,0,0,0,0,0,0,0,0,0,0,0,0
1,Fungal infection,skin_rash,nodal_skin_eruptions,dischromic _patches,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,Fungal infection,itching,nodal_skin_eruptions,dischromic _patches,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,Fungal infection,itching,skin_rash,dischromic _patches,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,Fungal infection,itching,skin_rash,nodal_skin_eruptions,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4915,(vertigo) Paroymsal Positional Vertigo,vomiting,headache,nausea,spinning_movements,loss_of_balance,unsteadiness,0,0,0,0,0,0,0,0,0,0,0
4916,Acne,skin_rash,pus_filled_pimples,blackheads,scurring,0,0,0,0,0,0,0,0,0,0,0,0,0
4917,Urinary tract infection,burning_micturition,bladder_discomfort,foul_smell_of urine,continuous_feel_of_urine,0,0,0,0,0,0,0,0,0,0,0,0,0
4918,Psoriasis,skin_rash,joint_pain,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,0,0,0,0,0,0,0,0,0,0,0


In [64]:
def find_diseases_for_symptom_pairs(df, symptom_pairs):

    disease_symptom_mapping = {}
    
    for symptom_pair in symptom_pairs:
        symptom1, symptom2 = symptom_pair
        
        # Find diseases that have both symptoms
        matching_diseases = []
        for _, row in df.iterrows():
            disease = row['Disease']
            symptoms = set(row[1:]) - {0}
            
            if symptom1 in symptoms and symptom2 in symptoms:
                matching_diseases.append(disease)
        
        if matching_diseases:
            disease_symptom_mapping[f"{symptom1} & {symptom2}"] = list(set(matching_diseases))
    
    return disease_symptom_mapping

# Find associated diseases
disease_mapping = find_diseases_for_symptom_pairs(df_clean, unique_itemsets)

print("Symptom Pairs and Associated Diseases:")
print("=" * 60)
for symptom_pair, diseases in disease_mapping.items():
    print(f"\n{symptom_pair}:")
    for disease in diseases:
        print(f"  - {disease}")

Symptom Pairs and Associated Diseases:

nausea & vomiting:
  - Chronic cholestasis
  - (vertigo) Paroymsal  Positional Vertigo
  - Typhoid
  - hepatitis A
  - Hypoglycemia
  - Malaria
  - Hepatitis D
  - Hepatitis E
  - Dengue

loss_of_appetite & yellowing_of_eyes:
  - Chronic cholestasis
  - Hepatitis B
  - hepatitis A
  - Hepatitis C
  - Tuberculosis
  - Hepatitis D
  - Hepatitis E

fatigue & loss_of_appetite:
  - Hepatitis B
  - Hepatitis C
  - Tuberculosis
  - Hepatitis D
  - Hepatitis E
  - Dengue
  - Chicken pox

loss_of_appetite & vomiting:
  - Chronic cholestasis
  - hepatitis A
  - Tuberculosis
  - Hepatitis D
  - Hepatitis E
  - Peptic ulcer diseae
  - Dengue

stomach_pain & yellowish_skin:
  - Chronic cholestasis
  - Hepatitis B
  - hepatitis A
  - Jaundice
  - Hepatitis D
  - Alcoholic hepatitis
  - Hepatitis E

stomach_pain & vomiting:
  - Chronic cholestasis
  - Typhoid
  - hepatitis A
  - Jaundice
  - Hepatitis D
  - GERD
  - Peptic ulcer diseae
  - Alcoholic hepatitis
 